# 🧪 Build RAGAS Evaluation Testset

This notebook generates a question-answer evaluation dataset from your uploaded documents.

It queries your RAG API to fetch document chunks, then uses an LLM to generate Q&A pairs directly — no fragile knowledge graph pipeline needed.

Run it **once** after uploading and indexing your documents.

In [ ]:
!uv pip install langchain-openai python-dotenv requests pandas tqdm

## ⚙️ Configuration

Edit `PROJECT_ID` and `BASE_URL` to match your running FastAPI instance.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path=os.path.join(os.path.dirname(os.getcwd()), 'src', '.env'))

# ── Edit these ──────────────────────────────────────────────
PROJECT_ID = 40
BASE_URL = "http://localhost:5000"
TESTSET_SIZE = 500
OUTPUT_PATH = "eval_results/testset.csv"
# ────────────────────────────────────────────────────────────

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
assert OPENAI_API_KEY, "OPENAI_API_KEY not found in src/.env!"

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
assert GROQ_API_KEY, "GROQ_API_KEY not found in src/.env!"

print(f"✅ Config loaded. Project ID: {PROJECT_ID}")

✅ Config loaded. Project ID: 40


## 📥 Step 1: Fetch Chunks & Merge into Long Documents

We pull raw text chunks from the API and merge every 5 consecutive chunks into longer documents.

In [2]:
import requests
from tqdm.auto import tqdm

def fetch_and_merge_chunks(project_id, base_url, queries, limit=20, chunks_per_doc=5, min_words=80):
    raw_texts = []
    seen = set()

    for query in queries:
        try:
            resp = requests.post(
                f"{base_url}/api/v1/nlp/index/search/{project_id}",
                json={"text": query, "limit": limit}, timeout=30,
            )
        except Exception as e:
            print(f"⚠️  Connection error for '{query}': {e}")
            continue
        if resp.status_code != 200:
            print(f"⚠️  Query '{query}' returned {resp.status_code}")
            continue
        for r in resp.json().get("results", []):
            text = r.get("text", "").strip()
            if text and text not in seen:
                seen.add(text)
                raw_texts.append(text)

    print(f"📄 Fetched {len(raw_texts)} unique raw chunks from {len(queries)} queries")

    merged = []
    for i in range(0, len(raw_texts), chunks_per_doc):
        group = raw_texts[i : i + chunks_per_doc]
        combined = "\n\n".join(group)
        if len(combined.split()) >= min_words:
            merged.append(combined)

    print(f"📦 Merged into {len(merged)} documents (groups of {chunks_per_doc})")
    if merged:
        avg = sum(len(d.split()) for d in merged) / len(merged)
        print(f"📊 Average words per document: {avg:.0f}")
    return merged

BROAD_QUERIES = [
    "introduction and overview", "main concepts and definitions",
    "methodology and approach", "results and findings",
    "challenges and limitations", "tools and technologies",
    "architecture and design", "evaluation and metrics",
    "deployment and production", "future work and conclusion",
]

docs = fetch_and_merge_chunks(PROJECT_ID, BASE_URL, BROAD_QUERIES)
print(f"\n✅ Total documents ready: {len(docs)}")

/home/ashraf/me/projects/grad_project/uniAct-rag-app/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


📄 Fetched 135 unique raw chunks from 10 queries
📦 Merged into 27 documents (groups of 5)
📊 Average words per document: 237

✅ Total documents ready: 27


## 🤖 Step 2: Generate Q&A Pairs

For each document, we ask the LLM to generate 2 question-answer pairs directly.
This is much more reliable than RAGAS's knowledge graph pipeline.

In [3]:
import json
import time
from openai import OpenAI

client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1",
)

SYSTEM_PROMPT = """You are a QA dataset generator. Given a document chunk, generate exactly 2 diverse question-answer pairs that can be answered using ONLY the information in the provided text.

Rules:
- Questions should be specific and varied (mix of what/how/why/when)
- Answers must be directly supported by the text (2-4 sentences each)
- Do NOT generate generic or vague questions
- Return ONLY valid JSON, no other text

Return format:
[
  {"question": "...", "answer": "..."},
  {"question": "...", "answer": "..."}
]"""

all_pairs = []
errors = 0
pairs_per_doc = 2

# Calculate how many docs we need to process
docs_needed = min(len(docs), (TESTSET_SIZE + pairs_per_doc - 1) // pairs_per_doc)

print(f"🚀 Generating Q&A pairs from {docs_needed} documents...")
print(f"   Target: {TESTSET_SIZE} pairs ({pairs_per_doc} per doc)")
print(f"   Using: llama-3.3-70b-versatile (free via Groq)\n")

for i, doc in enumerate(tqdm(docs[:docs_needed], desc="Generating Q&A")):
    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"Generate 2 question-answer pairs from this text:\n\n{doc}"}
            ],
            temperature=0.7,
            max_tokens=1000,
        )
        
        text = response.choices[0].message.content.strip()
        
        # Extract JSON from response (handle markdown code blocks)
        if "```" in text:
            text = text.split("```")[1]
            if text.startswith("json"):
                text = text[4:]
            text = text.strip()
        
        pairs = json.loads(text)
        
        for pair in pairs:
            all_pairs.append({
                "user_input": pair["question"],
                "reference": pair["answer"],
                "reference_contexts": doc,
            })
        
    except json.JSONDecodeError:
        errors += 1
        tqdm.write(f"⚠️ JSON parse error on doc {i}, skipping")
    except Exception as e:
        errors += 1
        error_msg = str(e)
        if "429" in error_msg:
            tqdm.write(f"⏳ Rate limited, waiting 30s...")
            time.sleep(30)
            # Retry once
            try:
                response = client.chat.completions.create(
                    model="llama-3.3-70b-versatile",
                    messages=[
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user", "content": f"Generate 2 question-answer pairs from this text:\n\n{doc}"}
                    ],
                    temperature=0.7, max_tokens=1000,
                )
                text = response.choices[0].message.content.strip()
                if "```" in text:
                    text = text.split("```")[1]
                    if text.startswith("json"):
                        text = text[4:]
                    text = text.strip()
                pairs = json.loads(text)
                for pair in pairs:
                    all_pairs.append({
                        "user_input": pair["question"],
                        "reference": pair["answer"],
                        "reference_contexts": doc,
                    })
            except:
                errors += 1
                tqdm.write(f"⚠️ Retry also failed on doc {i}")
        else:
            tqdm.write(f"⚠️ Error on doc {i}: {error_msg[:80]}")
    
    # Small delay to avoid rate limits
    time.sleep(1)

# Trim to exact size
all_pairs = all_pairs[:TESTSET_SIZE]

print(f"\n✅ Generated {len(all_pairs)} Q&A pairs!")
if errors:
    print(f"⚠️ {errors} documents had errors (skipped)")

🚀 Generating Q&A pairs from 27 documents...
   Target: 500 pairs (2 per doc)
   Using: llama-3.3-70b-versatile (free via Groq)



Generating Q&A: 100%|██████████| 27/27 [00:41<00:00,  1.53s/it]


✅ Generated 54 Q&A pairs!


## 💾 Step 3: Preview and Save the Testset

In [4]:
import os
import pandas as pd

df = pd.DataFrame(all_pairs)

print("📋 Testset Preview:")
print(df[["user_input", "reference"]].head(5).to_string(max_colwidth=80))
print(f"\nTotal samples: {len(df)}")
print(f"Columns: {list(df.columns)}")

os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
df.to_csv(OUTPUT_PATH, index=False)
print(f"\n✅ Testset saved to: {OUTPUT_PATH}")

📋 Testset Preview:
                                                                        user_input                                                                        reference
0        What types of data sources are covered in the provided learning material?  The learning material covers working with data from CSV, JSON, SQL, database...
1     How long has O'Reilly Media been providing technology and business training?  O'Reilly Media has provided technology and business training, knowledge, and...
2                                When was the second edition of the book released?  The second edition of the book was released in July 2023, with the first rel...
3  What is the purpose of using pretrained classifiers in machine learning pipe...  The purpose of using pretrained classifiers in machine learning pipelines is...
4         What are the two preprocessing steps contained in the preprocess object?  The two preprocessing steps contained in the preprocess object are StandardS.

## ✅ Done!

Your testset has been saved. Now run `ragas_evaluation.ipynb` to evaluate your RAG system using this testset.

The testset CSV has these columns:
- `user_input` — the generated question
- `reference` — the expected/reference answer
- `reference_contexts` — the source context used to create the question